# M-RE Inference on TEST SET — R1 (A5-fullctx) + R2 (A5-large)

Runs the full pairwise RE inference pipeline on `articles_test.json` for **two runs**.
Entity mentions come from the NER ensemble predictions (T611-R1).
Per-predicate thresholds are hardcoded from dev tuning — no re-tuning needed.
No evaluation (no ground truth on test).

| Run | Model folder | Window | Dev Micro-F1 |
|-----|-------------|--------|-------------|
| R1 | `pubmedbert_large_re_A6_fullctx` | full ctx | 0.5961 |
| R2 | `pubmedbert_large_re_A5_hardneg` | ±300 chars | 0.5932 |

Output folders (inside `src/re/predictions/`):
- `SMTE_T621_R1_REfullctx/`
- `SMTE_T621_R2_RElargewindow/`

## 0. Imports & paths — ⚠️ change only TEAM_ID if needed

In [1]:
from pathlib import Path
import os, json, re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

def find_repo_root(start):
    for p in [start] + list(start.parents):
        if (p / 'data').exists() and (p / 'src').exists():
            return p
    raise FileNotFoundError('Cannot find repo root')

PROJECT_ROOT = find_repo_root(Path.cwd())

# ── Shared inputs ─────────────────────────────────────────────────────────────
TEST_ARTICLES_PATH = (
    PROJECT_ROOT / 'data' / 'GutBrainIE_Full_Collection_2026'
    / 'Articles' / 'json_format' / 'articles_test.json'
)
NER_PRED_PATH = (
    PROJECT_ROOT / 'src' / 'ner' / 'predictions' / 'test_set'
    / 'SMTE_T611_R1_NERensemble' / 'SMTE_T611_R1_NERensemble.json'
)
BASE_MODEL   = 'microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract'
POOLING_MODE = 'mention-mean'  # same for both runs
BATCH_SIZE   = 16
MAX_LENGTH   = 512

TEAM_ID = 'SMTE'   # ← your CLEF 2026 team ID
TASK_ID = 'T621'

# ── Run definitions ───────────────────────────────────────────────────────────
RUNS = [
    {
        'run_id':       'R1',
        'system_desc':  'REfullctx',
        'model_dir':    PROJECT_ROOT / 'src' / 're' / 'models' / 'pubmedbert_large_re_A6_fullctx',
        'window_chars': 999999,   # full context — relies on 512-token truncation
        'dev_macro':    '0.5149',
        'dev_micro':    '0.5961',
        'desc_window':  'Full title+abstract context (A5-fullctx); 512-token truncation limit.',
    },
    {
        'run_id':       'R2',
        'system_desc':  'RElargewindow',
        'model_dir':    PROJECT_ROOT / 'src' / 're' / 'models' / 'pubmedbert_large_re_A5_hardneg',
        'window_chars': 300,      # ±300 chars around entity pair
        'dev_macro':    '0.5155',
        'dev_micro':    '0.5932',
        'desc_window':  'Local context window of +-300 characters around the entity pair (A5-large).',
    },
]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('PROJECT_ROOT :', PROJECT_ROOT)
print('DEVICE       :', DEVICE)
print('TEST_ARTICLES:', TEST_ARTICLES_PATH.exists())
print('NER_PRED     :', NER_PRED_PATH.exists())
print()
for r in RUNS:
    status = '✓' if r['model_dir'].exists() else '✗ MISSING'
    print(f"  {status}  {r['run_id']}  {r['model_dir'].name}")

C:\Users\super\Documents\UniPd\ATA\SMTE-GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT : C:\Users\super\Documents\UniPd\ATA\SMTE-GutBrainIE
DEVICE       : cuda
TEST_ARTICLES: True
NER_PRED     : True

  ✓  R1  pubmedbert_large_re_A6_fullctx
  ✓  R2  pubmedbert_large_re_A5_hardneg


## 1. Labels and legal relation pairs

In [2]:
LEGAL_RELATION_LABELS = {
    'administered','affect','change abundance','change effect','change expression','compared to',
    'impact','influence','interact','is a','is linked to','located in','part of','produced by',
    'strike','target','used by'
}
RELATION_LABELS = [
    'no relation','administered','affect','change abundance','change effect','change expression',
    'compared to','impact','influence','interact','is a','is linked to','located in','part of',
    'produced by','strike','target','used by'
]
label2id = {l: i for i, l in enumerate(RELATION_LABELS)}
id2label = {i: l for i, l in enumerate(RELATION_LABELS)}

def norm_ent(label):
    if label is None: return ''
    lab = str(label).strip()
    return 'DDF' if lab.lower() == 'ddf' else lab

def norm_span(s):
    return re.sub(r'\s+', ' ', str(s).strip())

LEGAL_RELATIONS = [
    ('DDF','affect','DDF'),('microbiome','is linked to','DDF'),('DDF','target','human'),
    ('drug','change effect','DDF'),('DDF','is a','DDF'),('microbiome','located in','human'),
    ('chemical','influence','DDF'),('dietary supplement','influence','DDF'),('DDF','target','animal'),
    ('chemical','impact','microbiome'),('anatomical location','located in','animal'),
    ('microbiome','located in','animal'),('chemical','located in','anatomical location'),
    ('bacteria','part of','microbiome'),('DDF','strike','anatomical location'),
    ('drug','administered','animal'),('bacteria','influence','DDF'),('drug','impact','microbiome'),
    ('DDF','change abundance','microbiome'),('microbiome','located in','anatomical location'),
    ('microbiome','used by','biomedical technique'),('chemical','produced by','microbiome'),
    ('dietary supplement','impact','microbiome'),('bacteria','located in','animal'),
    ('animal','used by','biomedical technique'),('chemical','impact','bacteria'),
    ('chemical','located in','animal'),('food','impact','bacteria'),
    ('microbiome','compared to','microbiome'),('human','used by','biomedical technique'),
    ('bacteria','change expression','gene'),('chemical','located in','human'),
    ('drug','interact','chemical'),('food','administered','human'),
    ('DDF','change abundance','bacteria'),('chemical','interact','chemical'),
    ('chemical','part of','chemical'),('dietary supplement','impact','bacteria'),
    ('DDF','interact','chemical'),('food','impact','microbiome'),('food','influence','DDF'),
    ('bacteria','located in','human'),('dietary supplement','administered','human'),
    ('bacteria','interact','chemical'),('drug','change expression','gene'),
    ('drug','impact','bacteria'),('drug','administered','human'),
    ('anatomical location','located in','human'),('dietary supplement','change expression','gene'),
    ('chemical','change expression','gene'),('bacteria','interact','bacteria'),
    ('drug','interact','drug'),('microbiome','change expression','gene'),
    ('bacteria','interact','drug'),('food','change expression','gene'),
]
legal_pairs = {}
for s, p, o in LEGAL_RELATIONS:
    legal_pairs.setdefault((norm_ent(s), norm_ent(o)), set()).add(p)

print(f'Relation labels : {len(RELATION_LABELS)}')
print(f'Legal type pairs: {len(legal_pairs)}')

Relation labels : 18
Legal type pairs: 52


## 2. Model architecture

In [3]:
def entity_average(hidden, mask):
    mask = mask.unsqueeze(-1).float()
    summed = (hidden * mask).sum(dim=1)
    count  = mask.sum(dim=1).clamp(min=1e-6)
    return summed / count

class BertForREWithEntityMarkers(nn.Module):
    def __init__(self, model_name, num_labels, pooling_mode='mention-mean'):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size * 2, num_labels)
        self.num_labels   = num_labels
        self.pooling_mode = pooling_mode

    def forward(self, input_ids, attention_mask, e1_mask, e2_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq = outputs.last_hidden_state
        if self.pooling_mode == 'mention-mean':
            e1_h = entity_average(seq, e1_mask)
            e2_h = entity_average(seq, e2_mask)
        else:
            e1_h = torch.bmm(e1_mask.unsqueeze(1).float(), seq).squeeze(1)
            e2_h = torch.bmm(e2_mask.unsqueeze(1).float(), seq).squeeze(1)
        concat_h = self.dropout(torch.cat([e1_h, e2_h], dim=-1))
        return {'logits': self.classifier(concat_h)}

print(f'Model class defined — pooling: {POOLING_MODE}')

Model class defined — pooling: mention-mean


## 3. Helper functions

In [4]:
def find_last_checkpoint(root_dir):
    ckpts = [d for d in root_dir.iterdir() if d.is_dir() and d.name.startswith('checkpoint-')]
    if not ckpts: return root_dir
    return sorted(ckpts, key=lambda x: int(x.name.split('-')[1]))[-1]

def create_full_text_with_offsets(title, abstract):
    return f'{title} {abstract}', len(title) + 1

def adjust_entity_positions(entity, abstract_offset):
    if entity['location'] == 'abstract':
        return {**entity,
                'start_idx': entity['start_idx'] + abstract_offset,
                'end_idx':   entity['end_idx']   + abstract_offset}
    return dict(entity)

def insert_entity_markers(text, subject, obj):
    entities = sorted([
        (subject['start_idx'], subject['end_idx'], '[E1]', '[/E1]'),
        (obj['start_idx'],     obj['end_idx'],     '[E2]', '[/E2]'),
    ], key=lambda x: x[0])
    marked, offset = text, 0
    for start, end, sm, em in entities:
        a, b = start + offset, end + offset + 1
        marked = marked[:a] + sm + marked[a:b] + em + marked[b:]
        offset += len(sm) + len(em)
    return marked

def build_window_around_entities(text, subject, obj, window_chars=300):
    left  = min(subject['start_idx'], obj['start_idx'])
    right = max(subject['end_idx'],   obj['end_idx'])
    ws = max(0, left - window_chars)
    we = min(len(text) - 1, right + window_chars)
    win = text[ws:we + 1]
    sw = {**subject, 'start_idx': subject['start_idx'] - ws, 'end_idx': subject['end_idx'] - ws}
    ow = {**obj,     'start_idx': obj['start_idx'] - ws,     'end_idx': obj['end_idx'] - ws}
    return win, sw, ow

def build_marked_text(text, subj, obj, window_chars=300):
    w, sw, ow = build_window_around_entities(text, subj, obj, window_chars)
    return insert_entity_markers(w, sw, ow)

print('Helper functions defined.')

Helper functions defined.


## 4. Decoding functions

In [5]:
PREDICATE_PRIOR = {'strike': 0.85, 'used by': 0.90, 'part of': 0.90}

def apply_predicate_prior(pred, prob):
    return prob * PREDICATE_PRIOR.get(pred, 1.0)

def softmax_np(x):
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / ex.sum()

def distance_min_prob(distance):
    if distance < 150:   return 0.10
    elif distance < 300: return 0.20
    else:                return 0.30

def row_to_legal_scores(row, label2id, id2label, temperature=1.0, renorm_legal=True):
    s_lab, o_lab = row['subject_label'], row['object_label']
    allowed_preds = sorted(legal_pairs.get((s_lab, o_lab), []))
    if not allowed_preds: return None
    logits = row['logits'].numpy() / temperature
    if renorm_legal:
        allowed_ids = [label2id['no relation']] + [label2id[p] for p in allowed_preds]
        probs_sub  = softmax_np(logits[allowed_ids])
        p_no       = float(probs_sub[0])
        rel_probs  = probs_sub[1:]
        best_idx   = int(np.argmax(rel_probs))
        pred_label = allowed_preds[best_idx]
        p_best     = float(rel_probs[best_idx])
        p_second   = sorted(rel_probs.tolist(), reverse=True)[1] if len(rel_probs) > 1 else 0.0
    else:
        probs = softmax_np(logits)
        p_no  = float(probs[label2id['no relation']])
        lp    = sorted([(id2label[label2id[p]], float(probs[label2id[p]])) for p in allowed_preds],
                        key=lambda x: x[1], reverse=True)
        pred_label, p_best = lp[0]
        p_second = lp[1][1] if len(lp) > 1 else 0.0
    p_best_adj = apply_predicate_prior(pred_label, p_best)
    return {'pred_label': pred_label, 'p_best_adj': p_best_adj,
            'margin_val': p_best - p_no, 'top2_gap': p_best - p_second}

def decode_doc(rows, default_max_chars, default_min_prob, default_margin, default_top2_gap,
               temperature, renorm_legal,
               max_chars_by_pred, min_prob_by_pred, margin_by_pred, top2_gap_by_pred,
               use_distance_aware_prob=True):
    best_for_pair = {}
    for row in rows:
        scores = row_to_legal_scores(row, label2id, id2label, temperature, renorm_legal)
        if scores is None: continue
        pred  = scores['pred_label']
        maxc  = max_chars_by_pred.get(pred,  default_max_chars)
        minp  = min_prob_by_pred.get(pred,   default_min_prob)
        marg  = margin_by_pred.get(pred,     default_margin)
        top2g = top2_gap_by_pred.get(pred,   default_top2_gap)
        if row['dist'] > maxc: continue
        eff_minp = max(minp, distance_min_prob(row['dist'])) if use_distance_aware_prob else minp
        if scores['p_best_adj'] < eff_minp:  continue
        if scores['margin_val'] < marg:      continue
        if scores['top2_gap']   < top2g:     continue
        k = row['k']
        prev = best_for_pair.get(k)
        if prev is None or scores['p_best_adj'] > prev[1]:
            best_for_pair[k] = (pred, scores['p_best_adj'])
    return [{'subject_text_span': st, 'subject_label': sl, 'predicate': pred,
             'object_text_span':  ot, 'object_label':  ol}
            for (st, sl, ot, ol), (pred, _) in sorted(best_for_pair.items())]

print('Decoding functions defined.')

Decoding functions defined.


## 5. Per-predicate thresholds

- **R1 (A5-fullctx)**: thresholds hardcoded from the dev tuning already run in `bert_RE_inference_unified.ipynb`.
- **R2 (A5-large)**: thresholds tuned on dev at runtime in section 5b below, using R2's own logit cache.

Both use the same global defaults and grids.

In [6]:
# ── Global defaults (shared) ──────────────────────────────────────────────────
BEST_MAX    = 250
BEST_MINP   = 0.10
BEST_MARG   = 0.25
BEST_TEMP   = 1.25
BEST_RENORM = True

# Tuning grids (used for R2 dev tuning)
PRED_MARGIN_GRID   = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
PRED_MINPROB_GRID  = [0.00, 0.10, 0.20, 0.25, 0.30, 0.35]
PRED_MAXCHARS_GRID = [100, 150, 200, 250, 300, 400, 500]
PRED_TOP2_GAP_GRID = [0.00, 0.02, 0.04, 0.06, 0.08]

# ── R1 thresholds — hardcoded from dev tuning on A5-fullctx ──────────────────
R1_THRESHOLDS = {
    'margin':   {
        'administered': 0.30, 'affect': 0.15, 'change abundance': 0.30,
        'change effect': 0.10, 'change expression': 0.10, 'compared to': 0.05,
        'impact': 0.10, 'influence': 0.30, 'interact': 0.15, 'is a': 0.10,
        'is linked to': 0.30, 'located in': 0.30, 'part of': 0.15,
        'produced by': 0.30, 'strike': 0.30, 'target': 0.10, 'used by': 0.30
    },
    'min_prob': {p: 0.0 for p in LEGAL_RELATION_LABELS},
    'max_chars': {
        'administered': 500, 'affect': 400, 'change abundance': 100,
        'change effect': 250, 'change expression': 250, 'compared to': 100,
        'impact': 200, 'influence': 250, 'interact': 150, 'is a': 100,
        'is linked to': 250, 'located in': 200, 'part of': 150,
        'produced by': 150, 'strike': 100, 'target': 200, 'used by': 150
    },
    'top2_gap': {p: 0.0 for p in LEGAL_RELATION_LABELS},
}

# R2 thresholds will be filled in section 5b after loading the dev cache
R2_THRESHOLDS = None

print('R1 thresholds ready. R2 will be tuned on dev in section 5b.')

R1 thresholds ready. R2 will be tuned on dev in section 5b.


## 5b. Tune R2 thresholds on dev (A5-large, window=300)

Loads R2 model, builds dev logit cache, runs Steps 1–4 threshold tuning, stores result in `R2_THRESHOLDS`.
GPU needed. Takes ~10 minutes.

In [7]:
# ── Load dev data ─────────────────────────────────────────────────────────────
DEV_PATH = (
    PROJECT_ROOT / 'data' / 'GutBrainIE_Full_Collection_2026'
    / 'Annotations' / 'Dev' / 'json_format' / 'dev.json'
)
with DEV_PATH.open(encoding='utf-8') as f:
    dev_data = json.load(f)

def build_gold_maps(dev_data):
    gold = {}
    for pmid, art in dev_data.items():
        s = set()
        for r in art.get('mention_level_relations', []):
            pred = r['predicate'].strip()
            if pred in LEGAL_RELATION_LABELS:
                s.add((
                    norm_span(r['subject_text_span']), norm_ent(r['subject_label']),
                    pred,
                    norm_span(r['object_text_span']),  norm_ent(r['object_label'])
                ))
        gold[str(pmid)] = s
    return gold

def micro_f1_for_pred(gold_by_doc, pred_by_doc, predicate):
    tp = fp = fn = 0
    for pmid, g in gold_by_doc.items():
        gp = {t for t in g if t[2] == predicate}
        pp = {t for t in pred_by_doc.get(pmid, set()) if t[2] == predicate}
        tp += len(gp & pp); fp += len(pp - gp); fn += len(gp - pp)
    P  = tp / (tp + fp) if (tp + fp) else 0.0
    R  = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * P * R / (P + R) if (P + R) else 0.0

gold_by_doc = build_gold_maps(dev_data)
PREDICATES  = sorted(LEGAL_RELATION_LABELS)
print(f'Dev docs: {len(dev_data)}  |  Gold relations: {sum(len(v) for v in gold_by_doc.values())}')

Dev docs: 80  |  Gold relations: 1139


In [8]:
# ── Load R2 model and build dev logit cache ───────────────────────────────────
R2_RUN    = RUNS[1]   # A5-large, window=300
R2_MODEL  = R2_RUN['model_dir']
R2_WINDOW = R2_RUN['window_chars']  # 300

print(f'Loading R2 model: {R2_MODEL.name}')
r2_load_dir   = find_last_checkpoint(R2_MODEL)
r2_state_path = r2_load_dir / 'pytorch_model.bin'
print(f'Checkpoint: {r2_load_dir.name}')

r2_tokenizer = AutoTokenizer.from_pretrained(str(R2_MODEL), use_fast=True, local_files_only=True)
r2_e1_id = r2_tokenizer.convert_tokens_to_ids('[E1]')
r2_e2_id = r2_tokenizer.convert_tokens_to_ids('[E2]')

r2_model = BertForREWithEntityMarkers(BASE_MODEL, num_labels=len(RELATION_LABELS),
                                       pooling_mode=POOLING_MODE)
r2_model.bert.resize_token_embeddings(len(r2_tokenizer))
r2_model.load_state_dict(torch.load(r2_state_path, map_location='cpu'))
r2_model.to(DEVICE).eval()
print(f'R2 model loaded on {DEVICE}')

# Build dev cache for R2 (uses gold entities from dev_data, window=300)
R2_DEV_CACHE_PATH = str(R2_MODEL / 'dev_logits_cache_r2.pt')

@torch.no_grad()
def cache_dev_logits_r2(model, tokenizer, dev_data, legal_pairs,
                         e1_token_id, e2_token_id, device,
                         batch_size=16, max_length=512, window_chars=300, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        print(f'[cache] loading: {Path(cache_path).name}')
        return torch.load(cache_path, map_location='cpu')
    model.eval()
    cache = {}
    for pmid, article in tqdm(dev_data.items(), desc='Caching R2 dev logits'):
        title    = article['metadata']['title']
        abstract = article['metadata']['abstract']
        full_text, abstract_offset = f'{title} {abstract}', len(title) + 1
        adjusted = [{
            **adjust_entity_positions(e, abstract_offset),
            'label':     norm_ent(e['label']),
            'text_span': norm_span(e['text_span']),
        } for e in article.get('entities', [])]
        pair_examples, rows_meta = [], []
        for i, subj in enumerate(adjusted):
            for j, obj in enumerate(adjusted):
                if i == j: continue
                if (subj['label'], obj['label']) not in legal_pairs: continue
                pair_examples.append((full_text, subj, obj))
                rows_meta.append({
                    'k':             (subj['text_span'], subj['label'], obj['text_span'], obj['label']),
                    'dist':          abs(subj['start_idx'] - obj['start_idx']),
                    'subject_label': subj['label'],
                    'object_label':  obj['label'],
                })
        if not pair_examples:
            cache[str(pmid)] = []
            continue
        marked_texts = [build_marked_text(t, s, o, window_chars) for t, s, o in pair_examples]
        rows, idx = [], 0
        for start in range(0, len(marked_texts), batch_size):
            batch = marked_texts[start:start + batch_size]
            enc = tokenizer(batch, truncation=True, max_length=max_length,
                            padding=True, return_tensors='pt')
            iids = enc['input_ids'].to(device)
            amsk = enc['attention_mask'].to(device)
            e1m  = (iids == e1_token_id).long()
            e2m  = (iids == e2_token_id).long()
            logits = model(input_ids=iids, attention_mask=amsk,
                           e1_mask=e1m, e2_mask=e2m)['logits'].detach().cpu().float()
            for b in range(logits.shape[0]):
                row = dict(rows_meta[idx])
                row['logits'] = logits[b]
                rows.append(row)
                idx += 1
        cache[str(pmid)] = rows
    if cache_path:
        torch.save(cache, cache_path)
        print(f'[cache] saved: {Path(cache_path).name}')
    print(f'Dev cache — docs: {len(cache)}, pairs: {sum(len(v) for v in cache.values())}')
    return cache

r2_dev_cache = cache_dev_logits_r2(
    model=r2_model, tokenizer=r2_tokenizer, dev_data=dev_data,
    legal_pairs=legal_pairs, e1_token_id=r2_e1_id, e2_token_id=r2_e2_id,
    device=DEVICE, batch_size=BATCH_SIZE, max_length=MAX_LENGTH,
    window_chars=R2_WINDOW, cache_path=R2_DEV_CACHE_PATH,
)

# Free GPU — not needed for threshold tuning
del r2_model
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('GPU memory freed.')

Loading R2 model: pubmedbert_large_re_A5_hardneg
Checkpoint: checkpoint-15528


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11171.25it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


R2 model loaded on cuda
[cache] loading: dev_logits_cache_r2.pt
GPU memory freed.


In [9]:
# ── Steps 1–4: per-predicate threshold tuning for R2 (CPU) ───────────────────

def decode_doc_for_tuning(rows, margin_by_pred, minprob_by_pred, maxchars_by_pred, top2gap_by_pred):
    """Decode using current threshold dicts; returns set of (st,sl,pred,ot,ol) tuples."""
    best_for_pair = {}
    for row in rows:
        scores = row_to_legal_scores(row, label2id, id2label, BEST_TEMP, BEST_RENORM)
        if scores is None: continue
        pred  = scores['pred_label']
        maxc  = maxchars_by_pred.get(pred,  BEST_MAX)
        minp  = minprob_by_pred.get(pred,   BEST_MINP)
        marg  = margin_by_pred.get(pred,    BEST_MARG)
        top2g = top2gap_by_pred.get(pred,   0.0)
        if row['dist'] > maxc: continue
        eff_minp = max(minp, distance_min_prob(row['dist']))
        if scores['p_best_adj'] < eff_minp: continue
        if scores['margin_val'] < marg:     continue
        if scores['top2_gap']   < top2g:    continue
        k = row['k']
        prev = best_for_pair.get(k)
        if prev is None or scores['p_best_adj'] > prev[1]:
            best_for_pair[k] = (pred, scores['p_best_adj'])
    return {(st, sl, pred, ot, ol) for (st, sl, ot, ol), (pred, _) in best_for_pair.items()}

# Step 1 — margin
print('Step 1 — tuning margin...')
r2_margin = {}
for pred in tqdm(PREDICATES, desc='margin', leave=False):
    best_f1, best_m = -1, BEST_MARG
    for m in PRED_MARGIN_GRID:
        pred_by_doc = {pmid: {t for t in decode_doc_for_tuning(
            rows,
            margin_by_pred={pred: m},
            minprob_by_pred={},
            maxchars_by_pred={},
            top2gap_by_pred={},
        ) if t[2] == pred} for pmid, rows in r2_dev_cache.items()}
        f1 = micro_f1_for_pred(gold_by_doc, pred_by_doc, pred)
        if f1 > best_f1: best_f1, best_m = f1, m
    r2_margin[pred] = best_m
print('  margin:', r2_margin)

# Step 2 — min_prob
print('Step 2 — tuning min_prob...')
r2_minprob = {}
for pred in tqdm(PREDICATES, desc='min_prob', leave=False):
    best_f1, best_mp = -1, BEST_MINP
    for mp in PRED_MINPROB_GRID:
        pred_by_doc = {pmid: {t for t in decode_doc_for_tuning(
            rows,
            margin_by_pred=r2_margin,
            minprob_by_pred={pred: mp},
            maxchars_by_pred={},
            top2gap_by_pred={},
        ) if t[2] == pred} for pmid, rows in r2_dev_cache.items()}
        f1 = micro_f1_for_pred(gold_by_doc, pred_by_doc, pred)
        if f1 > best_f1: best_f1, best_mp = f1, mp
    r2_minprob[pred] = best_mp
print('  min_prob:', r2_minprob)

# Step 3 — max_chars
print('Step 3 — tuning max_chars...')
r2_maxchars = {}
for pred in tqdm(PREDICATES, desc='max_chars', leave=False):
    best_f1, best_mc = -1, BEST_MAX
    for mc in PRED_MAXCHARS_GRID:
        pred_by_doc = {pmid: {t for t in decode_doc_for_tuning(
            rows,
            margin_by_pred=r2_margin,
            minprob_by_pred=r2_minprob,
            maxchars_by_pred={pred: mc},
            top2gap_by_pred={},
        ) if t[2] == pred} for pmid, rows in r2_dev_cache.items()}
        f1 = micro_f1_for_pred(gold_by_doc, pred_by_doc, pred)
        if f1 > best_f1: best_f1, best_mc = f1, mc
    r2_maxchars[pred] = best_mc
print('  max_chars:', r2_maxchars)

# Step 4 — top2_gap
print('Step 4 — tuning top2_gap...')
r2_top2gap = {}
for pred in tqdm(PREDICATES, desc='top2_gap', leave=False):
    best_f1, best_tg = -1, 0.0
    for tg in PRED_TOP2_GAP_GRID:
        pred_by_doc = {pmid: {t for t in decode_doc_for_tuning(
            rows,
            margin_by_pred=r2_margin,
            minprob_by_pred=r2_minprob,
            maxchars_by_pred=r2_maxchars,
            top2gap_by_pred={pred: tg},
        ) if t[2] == pred} for pmid, rows in r2_dev_cache.items()}
        f1 = micro_f1_for_pred(gold_by_doc, pred_by_doc, pred)
        if f1 > best_f1: best_f1, best_tg = f1, tg
    r2_top2gap[pred] = best_tg
print('  top2_gap:', r2_top2gap)

R2_THRESHOLDS = {
    'margin':    r2_margin,
    'min_prob':  r2_minprob,
    'max_chars': r2_maxchars,
    'top2_gap':  r2_top2gap,
}
print('\n✓ R2 thresholds tuned and stored in R2_THRESHOLDS.')

Step 1 — tuning margin...


KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  EXP.2 — Decoding ablation for R1 (A5-fullctx) on the DEV set — TEST decoder.
#  Reproduces the submitted operating point (dev Micro-F1 ≈ 0.5961) and ablates
#  one decoding component at a time (Reviewer 1).
#  RUN cells 0-14 first (setup + dev load + gold). You may SKIP cells 15-16 (R2).
#  Self-contained: builds its own R1 dev logit cache; uses the TEST scoring core.
# ══════════════════════════════════════════════════════════════════════════════

R1_RUN    = RUNS[0]                       # A5-fullctx
R1_MODEL  = R1_RUN['model_dir']
R1_WINDOW = R1_RUN['window_chars']        # 999999 = full context (512-token truncation)
print(f"R1 run: {R1_RUN['system_desc']} | model={R1_MODEL.name} | window={R1_WINDOW} | paper dev_micro={R1_RUN['dev_micro']}")

# --- load R1 model (clone of the R2 loader in cell 15) ---
r1_load_dir   = find_last_checkpoint(R1_MODEL)
r1_state_path = r1_load_dir / 'pytorch_model.bin'
r1_tokenizer  = AutoTokenizer.from_pretrained(str(R1_MODEL), use_fast=True, local_files_only=True)
r1_e1_id = r1_tokenizer.convert_tokens_to_ids('[E1]')
r1_e2_id = r1_tokenizer.convert_tokens_to_ids('[E2]')
r1_model = BertForREWithEntityMarkers(BASE_MODEL, num_labels=len(RELATION_LABELS), pooling_mode=POOLING_MODE)
r1_model.bert.resize_token_embeddings(len(r1_tokenizer))
r1_model.load_state_dict(torch.load(r1_state_path, map_location='cpu'))
r1_model.to(DEVICE).eval()
print(f"R1 model loaded — checkpoint {r1_load_dir.name}")

# --- build R1 dev logit cache (self-contained copy of cache_dev_logits_r2) ---
@torch.no_grad()
def cache_dev_logits_r1(model, tokenizer, dev_data, legal_pairs,
                        e1_token_id, e2_token_id, device,
                        batch_size, max_length, window_chars, cache_path):
    if cache_path and os.path.exists(cache_path):
        print(f'[cache] loading: {Path(cache_path).name}')
        return torch.load(cache_path, map_location='cpu')
    model.eval(); cache = {}
    for pmid, article in tqdm(dev_data.items(), desc='Caching R1 dev logits'):
        title    = article['metadata']['title']
        abstract = article['metadata']['abstract']
        full_text, abstract_offset = f'{title} {abstract}', len(title) + 1
        adjusted = [{
            **adjust_entity_positions(e, abstract_offset),
            'label':     norm_ent(e['label']),
            'text_span': norm_span(e['text_span']),
        } for e in article.get('entities', [])]
        pair_examples, rows_meta = [], []
        for i, subj in enumerate(adjusted):
            for j, obj in enumerate(adjusted):
                if i == j: continue
                if (subj['label'], obj['label']) not in legal_pairs: continue
                pair_examples.append((full_text, subj, obj))
                rows_meta.append({
                    'k':    (subj['text_span'], subj['label'], obj['text_span'], obj['label']),
                    'dist': abs(subj['start_idx'] - obj['start_idx']),
                    'subject_label': subj['label'], 'object_label': obj['label'],
                })
        if not pair_examples:
            cache[str(pmid)] = []; continue
        marked_texts = [build_marked_text(t, s, o, window_chars) for t, s, o in pair_examples]
        rows, idx = [], 0
        for start in range(0, len(marked_texts), batch_size):
            batch = marked_texts[start:start + batch_size]
            enc = tokenizer(batch, truncation=True, max_length=max_length, padding=True, return_tensors='pt')
            iids = enc['input_ids'].to(device); amsk = enc['attention_mask'].to(device)
            e1m = (iids == e1_token_id).long(); e2m = (iids == e2_token_id).long()
            logits = model(input_ids=iids, attention_mask=amsk, e1_mask=e1m, e2_mask=e2m)['logits'].detach().cpu().float()
            for b in range(logits.shape[0]):
                row = dict(rows_meta[idx]); row['logits'] = logits[b]; rows.append(row); idx += 1
        cache[str(pmid)] = rows
    if cache_path:
        torch.save(cache, cache_path); print(f'[cache] saved: {Path(cache_path).name}')
    print(f'Dev cache — docs: {len(cache)}, pairs: {sum(len(v) for v in cache.values())}')
    return cache

r1_dev_cache = cache_dev_logits_r1(
    r1_model, r1_tokenizer, dev_data, legal_pairs, r1_e1_id, r1_e2_id, DEVICE,
    BATCH_SIZE, MAX_LENGTH, R1_WINDOW, str(R1_MODEL / 'dev_logits_cache_r1.pt'))
del r1_model
if torch.cuda.is_available(): torch.cuda.empty_cache()

# --- toggle-able decode (tuple output, identical scoring core to decode_doc_for_tuning) ---
def _decode_toggle(rows, TH, temperature, use_distance, per_pred):
    best = {}
    for row in rows:
        scores = row_to_legal_scores(row, label2id, id2label, temperature, BEST_RENORM)
        if scores is None: continue
        pred = scores['pred_label']
        if per_pred:
            maxc = TH['max_chars'].get(pred, BEST_MAX); minp = TH['min_prob'].get(pred, BEST_MINP)
            marg = TH['margin'].get(pred, BEST_MARG);   top2g = TH['top2_gap'].get(pred, 0.0)
        else:
            maxc, minp, marg, top2g = BEST_MAX, BEST_MINP, BEST_MARG, 0.0
        if row['dist'] > maxc: continue
        eff = max(minp, distance_min_prob(row['dist'])) if use_distance else minp
        if scores['p_best_adj'] < eff:   continue
        if scores['margin_val'] < marg:  continue
        if scores['top2_gap']   < top2g: continue
        k = row['k']; prev = best.get(k)
        if prev is None or scores['p_best_adj'] > prev[1]:
            best[k] = (pred, scores['p_best_adj'])
    return {(st, sl, pred, ot, ol) for (st, sl, ot, ol), (pred, _) in best.items()}

def _micro_all(gold_by_doc, pred_by_doc):
    tp = fp = fn = 0
    for pmid, g in gold_by_doc.items():
        p = pred_by_doc.get(pmid, set())
        tp += len(g & p); fp += len(p - g); fn += len(g - p)
    P = tp / (tp + fp) if (tp + fp) else 0.0
    R = tp / (tp + fn) if (tp + fn) else 0.0
    F = 2 * P * R / (P + R) if (P + R) else 0.0
    return P, R, F

_configs = [
    ("Full decoding pipeline",       dict(temperature=BEST_TEMP, use_distance=True,  per_pred=True)),
    ("  - temperature scaling",      dict(temperature=1.0,       use_distance=True,  per_pred=True)),
    ("  - distance-aware floor",     dict(temperature=BEST_TEMP, use_distance=False, per_pred=True)),
    ("  - per-predicate thresholds", dict(temperature=BEST_TEMP, use_distance=True,  per_pred=False)),
]
print(f"\n{'Configuration':34s} {'Mic-P':>7} {'Mic-R':>7} {'Mic-F1':>7}")
print("-" * 62)
_base = None
for _name, _kw in _configs:
    pred_by_doc = {pmid: _decode_toggle(rows, R1_THRESHOLDS, **_kw) for pmid, rows in r1_dev_cache.items()}
    P, R, F = _micro_all(gold_by_doc, pred_by_doc)
    if _base is None: _base = F; d = ""
    else: d = f"  (delta {F - _base:+.4f})"
    print(f"{_name:34s} {P:.4f} {R:.4f} {F:.4f}{d}")
print("\nCheck: 'Full decoding pipeline' Mic-F1 should be ~0.5961 (paper headline).")


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
#  EXP.8 — Oracle: M-RE on GOLD entities vs PREDICTED entities (Reviewer 2).
#  Same model / decoder / R1 thresholds; ONLY the entity source changes.
#  Quantifies how much NER error propagation costs the RE stage.
#  Run after cells 0-14 (the R1 threshold block must have run). Self-contained.
# ══════════════════════════════════════════════════════════════════════════════
import json
from pathlib import Path

# dev.json = gold entities + gold relations (already loaded as DEV_PATH)
GOLD_DEV_PATH = DEV_PATH
# >>> SET THIS to your dev_pred.json (dev docs with PREDICTED NER entities) <<<
PRED_DEV_PATH = DEV_PATH.parent / 'dev_pred.json'
# e.g. PRED_DEV_PATH = Path('/full/path/to/dev_pred.json')

gold_dev = json.load(open(GOLD_DEV_PATH, encoding='utf-8'))
pred_dev = json.load(open(PRED_DEV_PATH, encoding='utf-8'))
print(f"gold-entity docs: {len(gold_dev)} | predicted-entity docs: {len(pred_dev)}")

# gold relations = single source of truth (from dev.json)
gold_rel_by_doc = {}
for pmid, art in gold_dev.items():
    s = set()
    for r in art.get('mention_level_relations', []):
        pr = r['predicate'].strip()
        if pr in LEGAL_RELATION_LABELS:
            s.add((norm_span(r['subject_text_span']), norm_ent(r['subject_label']), pr,
                   norm_span(r['object_text_span']),  norm_ent(r['object_label'])))
    gold_rel_by_doc[str(pmid)] = s
print("gold relations:", sum(len(v) for v in gold_rel_by_doc.values()))

# load R1 model (A5-fullctx)
R1_RUN = RUNS[0]; R1_MODEL = R1_RUN['model_dir']; R1_WINDOW = R1_RUN['window_chars']
_ld  = find_last_checkpoint(R1_MODEL)
_tok = AutoTokenizer.from_pretrained(str(R1_MODEL), use_fast=True, local_files_only=True)
_e1  = _tok.convert_tokens_to_ids('[E1]'); _e2 = _tok.convert_tokens_to_ids('[E2]')
_m   = BertForREWithEntityMarkers(BASE_MODEL, num_labels=len(RELATION_LABELS), pooling_mode=POOLING_MODE)
_m.bert.resize_token_embeddings(len(_tok))
_m.load_state_dict(torch.load(_ld / 'pytorch_model.bin', map_location='cpu'))
_m.to(DEVICE).eval()

@torch.no_grad()
def _cache(data, cache_path):
    if cache_path and os.path.exists(cache_path):
        print(f'[cache] loading {Path(cache_path).name}'); return torch.load(cache_path, map_location='cpu')
    cache = {}
    for pmid, article in tqdm(data.items(), desc=f'cache {Path(cache_path).stem}', leave=False):
        title = article['metadata']['title']; abstract = article['metadata']['abstract']
        full_text, off = f'{title} {abstract}', len(title) + 1
        adjusted = [{**adjust_entity_positions(e, off), 'label': norm_ent(e['label']),
                     'text_span': norm_span(e['text_span'])} for e in article.get('entities', [])]
        pe, rm = [], []
        for i, subj in enumerate(adjusted):
            for j, obj in enumerate(adjusted):
                if i == j: continue
                if (subj['label'], obj['label']) not in legal_pairs: continue
                pe.append((full_text, subj, obj))
                rm.append({'k': (subj['text_span'], subj['label'], obj['text_span'], obj['label']),
                           'dist': abs(subj['start_idx'] - obj['start_idx']),
                           'subject_label': subj['label'], 'object_label': obj['label']})
        if not pe: cache[str(pmid)] = []; continue
        mt = [build_marked_text(t, s, o, R1_WINDOW) for t, s, o in pe]
        rows, idx = [], 0
        for st in range(0, len(mt), BATCH_SIZE):
            b = mt[st:st + BATCH_SIZE]
            enc = _tok(b, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt')
            iid = enc['input_ids'].to(DEVICE); am = enc['attention_mask'].to(DEVICE)
            lo = _m(input_ids=iid, attention_mask=am, e1_mask=(iid == _e1).long(),
                    e2_mask=(iid == _e2).long())['logits'].detach().cpu().float()
            for bi in range(lo.shape[0]):
                row = dict(rm[idx]); row['logits'] = lo[bi]; rows.append(row); idx += 1
        cache[str(pmid)] = rows
    if cache_path: torch.save(cache, cache_path)
    return cache

gold_cache = _cache(gold_dev, str(R1_MODEL / 'dev_cache_ORACLE_gold.pt'))
pred_cache = _cache(pred_dev, str(R1_MODEL / 'dev_cache_ORACLE_pred.pt'))
del _m
if torch.cuda.is_available(): torch.cuda.empty_cache()

def _decode(rows):
    best = {}
    for row in rows:
        sc = row_to_legal_scores(row, label2id, id2label, BEST_TEMP, BEST_RENORM)
        if sc is None: continue
        pred = sc['pred_label']
        maxc = R1_THRESHOLDS['max_chars'].get(pred, BEST_MAX); minp = R1_THRESHOLDS['min_prob'].get(pred, BEST_MINP)
        marg = R1_THRESHOLDS['margin'].get(pred, BEST_MARG);   top2g = R1_THRESHOLDS['top2_gap'].get(pred, 0.0)
        if row['dist'] > maxc: continue
        if sc['p_best_adj'] < max(minp, distance_min_prob(row['dist'])): continue
        if sc['margin_val'] < marg: continue
        if sc['top2_gap'] < top2g: continue
        k = row['k']; prev = best.get(k)
        if prev is None or sc['p_best_adj'] > prev[1]: best[k] = (pred, sc['p_best_adj'])
    return {(st, sl, pred, ot, ol) for (st, sl, ot, ol), (pred, _) in best.items()}

def _micro(cache):
    tp = fp = fn = 0
    for pmid, rows in cache.items():
        p = _decode(rows); g = gold_rel_by_doc.get(pmid, set())
        tp += len(g & p); fp += len(p - g); fn += len(g - p)
    P = tp / (tp + fp) if (tp + fp) else 0.0
    R = tp / (tp + fn) if (tp + fn) else 0.0
    return P, R, (2 * P * R / (P + R) if (P + R) else 0.0)

Pg, Rg, Fg = _micro(gold_cache)
Pp, Rp, Fp = _micro(pred_cache)
print("\n=== EXP.8 Oracle: M-RE by entity source (identical model / decoder / thresholds) ===")
print(f"{'Entity source':24s} {'Mic-P':>7} {'Mic-R':>7} {'Mic-F1':>7}")
print("-" * 50)
print(f"{'Gold entities (oracle)':24s} {Pg:.4f} {Rg:.4f} {Fg:.4f}")
print(f"{'Predicted entities':24s} {Pp:.4f} {Rp:.4f} {Fp:.4f}")
print(f"{'Gap = NER-error cost':24s} {'':7} {'':7} {Fg - Fp:+.4f}")
print("\n(Expected: gold >= predicted; the gap is what NER errors cost the RE stage.)")


gold-entity docs: 80 | predicted-entity docs: 80
gold relations: 1139


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 21723.22it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== EXP.8 Oracle: M-RE by entity source (identical model / decoder / thresholds) ===
Entity source              Mic-P   Mic-R  Mic-F1
--------------------------------------------------
Gold entities (oracle)   0.5669 0.5654 0.5662
Predicted entities       0.4219 0.4363 0.4290
Gap = NER-error cost                     +0.1372

(Expected: gold >= predicted; the gap is what NER errors cost the RE stage.)


## 6. Load test data + NER predictions

In [ ]:
with TEST_ARTICLES_PATH.open(encoding='utf-8') as f:
    test_articles = json.load(f)
with NER_PRED_PATH.open(encoding='utf-8') as f:
    ner_preds = json.load(f)

print(f'Test articles : {len(test_articles)}')
print(f'NER pred PMIDs: {len(ner_preds)}')

missing = set(str(k) for k in test_articles) - set(str(k) for k in ner_preds)
if missing:
    print(f'⚠️  {len(missing)} test PMIDs missing from NER predictions')
else:
    print('✓ All test PMIDs covered by NER predictions')

## 7. Inference loop — runs R1 then R2

Each run:
1. Loads its model
2. Builds (or loads from cache) the logit cache for test pairs
3. Decodes with per-predicate thresholds
4. Validates submission format
5. Saves `folder/folder.json` + `folder/folder.meta`

In [ ]:
@torch.no_grad()
def cache_logits(model, tokenizer, test_articles, ner_preds, legal_pairs,
                 e1_token_id, e2_token_id, device,
                 batch_size, max_length, window_chars, cache_path):
    if cache_path and os.path.exists(cache_path):
        print(f'  [cache] loading: {Path(cache_path).name}')
        return torch.load(cache_path, map_location='cpu')

    model.eval()
    cache, skipped = {}, 0

    for pmid, article in tqdm(test_articles.items(), desc='  Caching logits', leave=False):
        meta     = article.get('metadata', article)
        title    = (meta.get('title') or '').strip()
        abstract = (meta.get('abstract') or '').strip()
        full_text, abstract_offset = f'{title} {abstract}', len(title) + 1

        ner_entities = ner_preds.get(str(pmid), {}).get('entities', [])
        if not ner_entities:
            skipped += 1
            cache[str(pmid)] = []
            continue

        adjusted = [{
            **adjust_entity_positions(e, abstract_offset),
            'label':     norm_ent(e['label']),
            'text_span': norm_span(e['text_span']),
        } for e in ner_entities]

        pair_examples, rows_meta = [], []
        for i, subj in enumerate(adjusted):
            for j, obj in enumerate(adjusted):
                if i == j: continue
                if (subj['label'], obj['label']) not in legal_pairs: continue
                pair_examples.append((full_text, subj, obj))
                rows_meta.append({
                    'k':             (subj['text_span'], subj['label'], obj['text_span'], obj['label']),
                    'dist':          abs(subj['start_idx'] - obj['start_idx']),
                    'subject_label': subj['label'],
                    'object_label':  obj['label'],
                })

        if not pair_examples:
            cache[str(pmid)] = []
            continue

        marked_texts = [build_marked_text(t, s, o, window_chars) for t, s, o in pair_examples]
        rows, idx = [], 0
        for start in range(0, len(marked_texts), batch_size):
            batch = marked_texts[start:start + batch_size]
            enc = tokenizer(batch, truncation=True, max_length=max_length,
                            padding=True, return_tensors='pt')
            iids = enc['input_ids'].to(device)
            amsk = enc['attention_mask'].to(device)
            e1m  = (iids == e1_token_id).long()
            e2m  = (iids == e2_token_id).long()
            logits = model(input_ids=iids, attention_mask=amsk,
                           e1_mask=e1m, e2_mask=e2m)['logits'].detach().cpu().float()
            for b in range(logits.shape[0]):
                row = dict(rows_meta[idx])
                row['logits'] = logits[b]
                rows.append(row)
                idx += 1
        cache[str(pmid)] = rows

    torch.save(cache, cache_path)
    total_pairs = sum(len(v) for v in cache.values())
    print(f'  [cache] saved — docs: {len(cache)}, pairs: {total_pairs}')
    if skipped: print(f'  ⚠️  {skipped} articles had no NER entities')
    return cache


def validate_re(predictions):
    REQUIRED = {'subject_text_span','subject_label','predicate','object_text_span','object_label'}
    errors = []
    for pmid, obj in predictions.items():
        if set(obj.keys()) != {'mention_level_relations'}:
            errors.append(f'{pmid}: unexpected keys {set(obj.keys())}')
        for i, r in enumerate(obj['mention_level_relations']):
            missing = REQUIRED - set(r.keys())
            if missing: errors.append(f'{pmid}[{i}]: missing {missing}')
            if r.get('predicate') not in LEGAL_RELATION_LABELS:
                errors.append(f'{pmid}[{i}]: illegal predicate "{r.get("predicate")}"')
    return errors


def make_meta(run, team_id, task_id):
    folder_name = f"{team_id}_{task_id}_{run['run_id']}_{run['system_desc']}"
    return (
        f"Team ID:         {team_id}\n"
        f"Task ID:         {task_id}\n"
        f"Run ID:          {run['run_id']}\n"
        "\n"
        "Type of training:\n"
        "  Fine-tuning of PubMedBERT-large on GutBrainIE 2026 training data\n"
        "  with pairwise relation classification (18 classes: 17 predicates + no_relation).\n"
        "\n"
        "Pre-processing methods:\n"
        "  - Title and abstract concatenated; abstract offsets shifted by len(title)+1.\n"
        "  - Candidate pairs from all ordered (subject, object) entity pairs\n"
        "    in the legal-pair dictionary (52 type pairs, 55 patterns).\n"
        f"  - Context: {run['desc_window']}\n"
        "  - Entity markers [E1]...[/E1] and [E2]...[/E2] inserted at mention boundaries.\n"
        "  - Mention-mean pooling over entity span hidden states.\n"
        "  - Hard negative sampling: 70% hard negatives + 30% easy negatives.\n"
        "  - Legal-only decoding with temperature scaling (T=1.25).\n"
        "  - Distance-aware minimum probability floor.\n"
        "  - Per-predicate threshold tuning (margin, min_prob, max_chars, top2_gap) on dev set.\n"
        "\n"
        "Training data used:\n"
        "  Gold + Silver + Silver 2025 + Bronze (4921 documents total).\n"
        "\n"
        "Relevant details of the run:\n"
        f"  Model: PubMedBERT-large — folder: {run['model_dir'].name}\n"
        "  Entity mentions: from NER ensemble predictions (SMTE_T611_R1_NERensemble).\n"
        "  lr=1e-5, 2 epochs, effective batch size 32, bf16, gradient checkpointing.\n"
        f"  Dev results: Macro-F1={run['dev_macro']}, Micro-F1={run['dev_micro']}.\n"
        "\n"
        "GitHub repository:\n"
        "  https://github.com/TODO_INSERT_REPO_LINK\n"
    )


# ── Main loop ─────────────────────────────────────────────────────────────────
PRED_DIR = PROJECT_ROOT / 'src' / 're' / 'predictions' / 'test_set'
PRED_DIR.mkdir(parents=True, exist_ok=True)

all_results = {}

for run in RUNS:
    folder_name = f"{TEAM_ID}_{TASK_ID}_{run['run_id']}_{run['system_desc']}"
    print(f"\n{'='*60}")
    print(f"  {folder_name}")
    print(f"{'='*60}")

    if not run['model_dir'].exists():
        print(f'  ✗ SKIPPED — model folder not found: {run["model_dir"]}')
        continue

    # Load model
    print('  Loading model…')
    load_dir   = find_last_checkpoint(run['model_dir'])
    state_path = load_dir / 'pytorch_model.bin'
    print(f'  Checkpoint: {load_dir.name}')

    tokenizer = AutoTokenizer.from_pretrained(str(run['model_dir']),
                                              use_fast=True, local_files_only=True)
    e1_id = tokenizer.convert_tokens_to_ids('[E1]')
    e2_id = tokenizer.convert_tokens_to_ids('[E2]')

    model = BertForREWithEntityMarkers(BASE_MODEL, num_labels=len(RELATION_LABELS),
                                       pooling_mode=POOLING_MODE)
    model.bert.resize_token_embeddings(len(tokenizer))
    model.load_state_dict(torch.load(state_path, map_location='cpu'))
    model.to(DEVICE).eval()
    print(f'  Model loaded on {DEVICE}')

    # Build logit cache
    cache_path = str(run['model_dir'] / 'test_logits_cache.pt')
    test_cache = cache_logits(
        model=model, tokenizer=tokenizer,
        test_articles=test_articles, ner_preds=ner_preds,
        legal_pairs=legal_pairs,
        e1_token_id=e1_id, e2_token_id=e2_id,
        device=DEVICE, batch_size=BATCH_SIZE, max_length=MAX_LENGTH,
        window_chars=run['window_chars'], cache_path=cache_path,
    )

    # Free GPU memory before next model
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # Select per-run thresholds
    thr = R1_THRESHOLDS if run['run_id'] == 'R1' else R2_THRESHOLDS
    assert thr is not None, 'R2_THRESHOLDS not set — run section 5b first!'

    # Decode
    predictions = {}
    for pmid, rows in test_cache.items():
        rels = decode_doc(
            rows,
            default_max_chars=BEST_MAX, default_min_prob=BEST_MINP,
            default_margin=BEST_MARG,   default_top2_gap=0.0,
            temperature=BEST_TEMP,      renorm_legal=BEST_RENORM,
            max_chars_by_pred=thr['max_chars'],
            min_prob_by_pred=thr['min_prob'],
            margin_by_pred=thr['margin'],
            top2_gap_by_pred=thr['top2_gap'],
        )
        predictions[str(pmid)] = {'mention_level_relations': rels}

    total_rels = sum(len(v['mention_level_relations']) for v in predictions.values())
    print(f'  Predicted relations: {total_rels}')

    # Validate
    errors = validate_re(predictions)
    if errors:
        print(f'  ⚠️  {len(errors)} validation error(s):')
        for e in errors[:10]: print(f'    {e}')
        continue
    print('  ✓ Validation passed')

    # Save submission folder
    out_dir = PRED_DIR / folder_name
    out_dir.mkdir(parents=True, exist_ok=True)

    json_path = out_dir / f'{folder_name}.json'
    with json_path.open('w', encoding='utf-8') as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    print(f'  ✓ JSON : {json_path.name}')

    meta_path = out_dir / f'{folder_name}.meta'
    meta_path.write_text(make_meta(run, TEAM_ID, TASK_ID), encoding='utf-8')
    print(f'  ✓ META : {meta_path.name}')

    all_results[run['run_id']] = predictions

print('\n✓ All runs completed.')

## 8. Comparison stats: R1 vs R2

In [ ]:
print(f"{'Run':<6}  {'Articles':>8}  {'Relations':>10}  {'Predicates (top 5)'}")
print('-' * 70)

for run_id, preds in all_results.items():
    n_arts = len(preds)
    n_rels = sum(len(v['mention_level_relations']) for v in preds.values())
    pred_counts = Counter(
        r['predicate']
        for v in preds.values()
        for r in v['mention_level_relations']
    )
    top5 = ', '.join(f"{p}:{c}" for p, c in pred_counts.most_common(5))
    print(f"{run_id:<6}  {n_arts:>8}  {n_rels:>10}  {top5}")

print()
print('Per-predicate breakdown:')
all_preds = sorted(LEGAL_RELATION_LABELS)
header = f"{'Predicate':<25}" + ''.join(f"  {r['run_id']:>8}" for r in RUNS)
print(header)
print('-' * len(header))
for pred in sorted(all_preds):
    row = f"{pred:<25}"
    for run_id, preds in all_results.items():
        cnt = sum(1 for v in preds.values()
                  for r in v['mention_level_relations'] if r['predicate'] == pred)
        row += f"  {cnt:>8}"
    print(row)